In [1]:
# ================================
# exp27_xgboost_pca
# XGBoost with PCA compression
# ================================

import pandas as pd
import numpy as np
import os

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor


def metric_diagnostics(y_true,y_pred):

    rmse=np.sqrt(mean_squared_error(y_true,y_pred))
    mae=mean_absolute_error(y_true,y_pred)

    y_true_log=np.log1p(y_true)
    y_pred_log=np.log1p(np.maximum(y_pred,0))
    rmsle=np.sqrt(mean_squared_error(y_true_log,y_pred_log))

    nrmse_mean=rmse/np.mean(y_true)
    nrmse_range=rmse/(np.max(y_true)-np.min(y_true))

    print("\nMetric diagnostics")
    print("------------------")
    print("RMSE:",rmse)
    print("MAE:",mae)
    print("RMSLE:",rmsle)
    print("NRMSE (mean):",nrmse_mean)
    print("NRMSE (range):",nrmse_range)


train=pd.read_csv("../data/train.csv",encoding="cp932")
test=pd.read_csv("../data/test.csv",encoding="cp932")

target="含水率"
id_col="sample number"

spectral_cols=[c for c in train.columns if c not in ["sample number","species number","樹種","含水率"]]

X=train[spectral_cols]
y=train[target]
X_test=test[spectral_cols]


kf=KFold(n_splits=5,shuffle=True,random_state=42)

oof=np.zeros(len(X))
test_pred=np.zeros(len(X_test))

model=Pipeline([
("scaler",StandardScaler()),
("pca",PCA(n_components=120)),
("model",XGBRegressor(
    n_estimators=900,
    max_depth=4,
    learning_rate=0.04,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
))
])

for fold,(train_idx,val_idx) in enumerate(kf.split(X)):

    X_train,X_val=X.iloc[train_idx],X.iloc[val_idx]
    y_train,y_val=y.iloc[train_idx],y.iloc[val_idx]

    model.fit(X_train,y_train)

    pred=model.predict(X_val)

    oof[val_idx]=pred
    test_pred+=model.predict(X_test)/kf.n_splits


print("\nFinal Model Performance")
metric_diagnostics(y,oof)


os.makedirs("../submissions",exist_ok=True)

submission=pd.DataFrame({
    id_col:test[id_col],
    target:test_pred
})

output_path="../submissions/exp27_xgboost_pca.csv"

submission.to_csv(output_path,index=False,header=False)

print("\nSubmission saved:",output_path)


Final Model Performance

Metric diagnostics
------------------
RMSE: 7.179979301673204
MAE: 3.263503595098084
RMSLE: 0.12622728790946586
NRMSE (mean): 0.14377881485288185
NRMSE (range): 0.02411483099723585

Submission saved: ../submissions/exp27_xgboost_pca.csv
